In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy import stats

# --- CONFIGURATION ---
# Ensure output directories exist
os.makedirs('../reports/figures', exist_ok=True)
FIG_DIR = '../reports/figures'

# Load Data
print("Loading customer features...")
df = pd.read_csv('../data/processed/customer_features.csv')
print(f"Data Loaded: {df.shape}")

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# --- 1. TARGET DISTRIBUTION (1 Plot) ---
plt.figure()
ax = sns.countplot(x='Churn', data=df, palette='viridis')
plt.title('Distribution of Churn (Target Variable)')
plt.xlabel('Churn (0=Active, 1=Churned)')
plt.ylabel('Count')
# Add percentages
total = len(df)
for p in ax.patches:
    percentage = f'{100 * p.get_height() / total:.1f}%'
    x = p.get_x() + p.get_width() / 2
    y = p.get_height()
    ax.annotate(percentage, (x, y), ha='center', va='bottom')
plt.savefig(f'{FIG_DIR}/churn_distribution.png')
plt.close()
print("Saved: churn_distribution.png")

# --- 2. RFM ANALYSIS (4 Plots) ---
rfm_cols = ['Recency', 'Frequency', 'TotalSpent', 'AvgTransactionValue']
for col in rfm_cols:
    plt.figure()
    sns.boxplot(x='Churn', y=col, data=df, showfliers=False, palette='Set2')
    plt.title(f'{col} Distribution by Churn Status')
    plt.yscale('log') # Log scale helps visualization
    plt.savefig(f'{FIG_DIR}/rfm_{col.lower()}_boxplot.png')
    plt.close()
    print(f"Saved: rfm_{col.lower()}_boxplot.png")

# --- 3. CORRELATION ANALYSIS (1 Plot) ---
plt.figure(figsize=(12, 10))
# Select numerical columns only
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()
# Mask upper triangle
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', vmax=.3, center=0,
            square=True, linewidths=.5, cbar_kws={"shrink": .5})
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/feature_correlation.png')
plt.close()
print("Saved: feature_correlation.png")

# --- 4. SEGMENT ANALYSIS (3 Plots) ---
# Churn Rate by Segment
if 'CustomerSegment' in df.columns:
    plt.figure()
    segment_churn = df.groupby('CustomerSegment')['Churn'].mean().sort_values()
    sns.barplot(x=segment_churn.index, y=segment_churn.values, palette='magma')
    plt.title('Churn Rate by Customer Segment')
    plt.ylabel('Churn Probability')
    plt.xticks(rotation=45)
    plt.savefig(f'{FIG_DIR}/segment_churn_rate.png')
    plt.close()
    print("Saved: segment_churn_rate.png")

# --- 5. TEMPORAL PATTERNS (2 Plots) ---
# Lateness Score vs Churn
plt.figure()
sns.kdeplot(data=df, x='LatenessScore', hue='Churn', fill=True, common_norm=False, palette='crest')
plt.title('Lateness Score Density (Active vs Churned)')
plt.xlim(0, 5) # Focus on meaningful range
plt.savefig(f'{FIG_DIR}/lateness_density.png')
plt.close()
print("Saved: lateness_density.png")

# --- 6. STATISTICAL TESTS ---
print("\n--- STATISTICAL SIGNIFICANCE (T-Tests) ---")
with open('../reports/eda_stats.txt', 'w') as f:
    f.write("T-Test Results (Comparing Active vs Churned Means)\n")
    f.write("="*50 + "\n")
    
    features_to_test = ['Recency', 'LatenessScore', 'TotalSpent', 'Frequency']
    churned = df[df['Churn'] == 1]
    active = df[df['Churn'] == 0]
    
    for feature in features_to_test:
        t_stat, p_val = stats.ttest_ind(churned[feature], active[feature], equal_var=False)
        result = "SIGNIFICANT" if p_val < 0.05 else "NOT SIGNIFICANT"
        line = f"{feature}: t={t_stat:.2f}, p={p_val:.4f} -> {result}\n"
        print(line.strip())
        f.write(line)

print("\nEDA Complete. Figures saved to reports/figures/")

Loading customer features...
Data Loaded: (3079, 17)
Saved: churn_distribution.png


/var/folders/_x/4d89ssgs22s77vcp80740swh0000gn/T/ipykernel_6750/55905971.py:24: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.countplot(x='Churn', data=df, palette='viridis')
/var/folders/_x/4d89ssgs22s77vcp80740swh0000gn/T/ipykernel_6750/55905971.py:43: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x='Churn', y=col, data=df, showfliers=False, palette='Set2')


Saved: rfm_recency_boxplot.png
Saved: rfm_frequency_boxplot.png


/var/folders/_x/4d89ssgs22s77vcp80740swh0000gn/T/ipykernel_6750/55905971.py:43: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x='Churn', y=col, data=df, showfliers=False, palette='Set2')
/var/folders/_x/4d89ssgs22s77vcp80740swh0000gn/T/ipykernel_6750/55905971.py:43: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x='Churn', y=col, data=df, showfliers=False, palette='Set2')


Saved: rfm_totalspent_boxplot.png
Saved: rfm_avgtransactionvalue_boxplot.png


/var/folders/_x/4d89ssgs22s77vcp80740swh0000gn/T/ipykernel_6750/55905971.py:43: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x='Churn', y=col, data=df, showfliers=False, palette='Set2')


Saved: feature_correlation.png
Saved: segment_churn_rate.png
Saved: lateness_density.png

--- STATISTICAL SIGNIFICANCE (T-Tests) ---
Recency: t=16.13, p=0.0000 -> SIGNIFICANT
LatenessScore: t=15.26, p=0.0000 -> SIGNIFICANT
TotalSpent: t=-14.13, p=0.0000 -> SIGNIFICANT
Frequency: t=-15.61, p=0.0000 -> SIGNIFICANT

EDA Complete. Figures saved to reports/figures/


/var/folders/_x/4d89ssgs22s77vcp80740swh0000gn/T/ipykernel_6750/55905971.py:70: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=segment_churn.index, y=segment_churn.values, palette='magma')
